In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, PCA
from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, DoubleType  # Add this import
from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from pyspark.sql.functions import udf
from pyspark.sql.types import LongType

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Jupyter Notebook with Increased Memory").config("spark.executor.memory", "8g").config("spark.driver.memory", "8g").config("spark.memory.fraction", "0.8").config("spark.executor.instances", "4").getOrCreate()


24/11/15 15:31:08 WARN Utils: Your hostname, Yashwanths-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.166.166.195 instead (on interface en0)
24/11/15 15:31:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/11/15 15:31:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
# Path to your Parquet file
parquet_file_path = "/Users/yashwanthp/Downloads/DAEN690_Data/indo_10m.parquet"

# Read Parquet file into DataFrame
df = spark.read.parquet(parquet_file_path)

# Show rows
df.show()

# Print the number of rows and columns
num_rows = df.count()  # Total number of rows
num_columns = len(df.columns)  # Total number of columns

print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_columns}")



24/11/14 11:27:06 WARN Utils: Your hostname, Yashwanths-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.201 instead (on interface en0)
24/11/14 11:27:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/11/14 11:27:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/11/14 11:27:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+--------------+---------------+--------------+---------------+---------------+----------+----------+----------------+----------------+-----------+-------------+------------------+---------------+-----------------+----------------+-----------------+-----------------------+-------------+-------------------+---------------+----------------+----------------+----------------------+----------------------+-------------------+---------+--------+----------+--------------+---------+-------+----------+-------+-----+-----+-----+---------+----------+-----------+-------+---------+------------------+-------------------+------------+---------+--------+----------+----------+-----------------+------------+----------------+---------------+--------------+------------+------------+----------+
|ZONE_ZONE_ID|ZONE_POLLER_ID|ZONE_FIRST_SEEN|ZONE_LAST_SEEN|      ZONE_NAME|ZONE_MON_REASON|ZONE_AVAIL|ZONE_VERIF|ZONE_VALID_DELEG|ZONE_VALID_FRESH|ZONE_DNSSEC|ZONE_STATS_ID|ZONE_STATS_ZONE_ID|ZONE_STATS

In [10]:
# Select specific columns from the dataset
selected_columns = [
    "NS_SEEN", "AVG_RTT_MILLIS", "IS_ONLINE", "IS_AUTH", "SERIAL", 
    "VERSION", "EDNS0", "CNAME", "BOGUS", "VALID_SOA", "KEYS_VERIF", "FOUND_WHERE", 
    "IP", "KS_SEEN", "KS_SUCCEEDED", "KS_SMALLEST_BUFF", 
    "KS_LARGEST_BUFF", "KS_LARGEST_MSG", "KS_TCP_AVAIL", "KS_SET_INCEP", "KS_SET_EXP"
]
data = df.select(*selected_columns)

# Display schema for validation
data.printSchema()

# Filter numerical columns (for feature engineering)
numerical_cols = [field.name for field in data.schema.fields if isinstance(field.dataType, (IntegerType, DoubleType))]
print(f"Numerical Columns: {numerical_cols}")

# Exclude 'IS_ONLINE' from feature columns
feature_cols = [col for col in numerical_cols if col != "IS_ONLINE"]
print(f"Feature Columns (excluding IS_ONLINE): {feature_cols}")


root
 |-- NS_SEEN: integer (nullable = true)
 |-- AVG_RTT_MILLIS: string (nullable = true)
 |-- IS_ONLINE: integer (nullable = true)
 |-- IS_AUTH: integer (nullable = true)
 |-- SERIAL: integer (nullable = true)
 |-- VERSION: string (nullable = true)
 |-- EDNS0: integer (nullable = true)
 |-- CNAME: integer (nullable = true)
 |-- BOGUS: integer (nullable = true)
 |-- VALID_SOA: integer (nullable = true)
 |-- KEYS_VERIF: integer (nullable = true)
 |-- FOUND_WHERE: integer (nullable = true)
 |-- IP: string (nullable = true)
 |-- KS_SEEN: integer (nullable = true)
 |-- KS_SUCCEEDED: integer (nullable = true)
 |-- KS_SMALLEST_BUFF: integer (nullable = true)
 |-- KS_LARGEST_BUFF: integer (nullable = true)
 |-- KS_LARGEST_MSG: integer (nullable = true)
 |-- KS_TCP_AVAIL: integer (nullable = true)
 |-- KS_SET_INCEP: integer (nullable = true)
 |-- KS_SET_EXP: integer (nullable = true)

Numerical Columns: ['NS_SEEN', 'IS_ONLINE', 'IS_AUTH', 'SERIAL', 'EDNS0', 'CNAME', 'BOGUS', 'VALID_SOA', 'KEY

In [11]:
# Check if feature_cols is empty
if not feature_cols:
    raise ValueError("No numerical features found for analysis.")

# Drop rows with missing values in feature columns
data = data.dropna(subset=feature_cols + ["IS_ONLINE"])

In [12]:
# # Assemble features for SVM model (original data)
# vector_assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
# data_with_features = vector_assembler.transform(data)

# # Check assembled features
# data_with_features.select("features").show(5)

# # Perform PCA to reduce dimensionality (for comparison)
# pca = PCA(k=5, inputCol="features", outputCol="pca_features")  # Keep top 5 components
# pca_model = pca.fit(data_with_features)
# result = pca_model.transform(data_with_features)


+--------------------+
|            features|
+--------------------+
|[1.504868557E9,1....|
|[1.504868557E9,1....|
|[1.504868557E9,1....|
|[1.504868557E9,1....|
|[1.504868557E9,1....|
+--------------------+
only showing top 5 rows



24/11/14 11:27:35 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/11/14 11:27:42 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [13]:
# # Assemble features for PCA data
# vector_assembler_pca = VectorAssembler(inputCols=["pca_features"], outputCol="pca_features_vector")
# data_with_pca_features = vector_assembler_pca.transform(result)

# # Split the data into training and test sets (80% train, 20% test)
# train_data, test_data = data_with_features.randomSplit([0.8, 0.2], seed=42)
# train_data_pca, test_data_pca = data_with_pca_features.randomSplit([0.8, 0.2], seed=42)

# # Print the size of the training and test sets
# print(f"Training set size (original): {train_data.count()}")
# print(f"Test set size (original): {test_data.count()}")
# print(f"Training set size (PCA): {train_data_pca.count()}")
# print(f"Test set size (PCA): {test_data_pca.count()}")

# # Initialize LinearSVC models for original and PCA data
# svm_original = LinearSVC(featuresCol="features", labelCol="IS_ONLINE")
# svm_pca = LinearSVC(featuresCol="pca_features_vector", labelCol="IS_ONLINE")

# # Train SVM on original data
# svm_model_original = svm_original.fit(train_data)

# # Train SVM on PCA data
# svm_model_pca = svm_pca.fit(train_data_pca)

# # Make predictions on test data
# predictions_original = svm_model_original.transform(test_data)
# predictions_pca = svm_model_pca.transform(test_data_pca)

# # Initialize evaluator for binary classification
# evaluator = BinaryClassificationEvaluator(labelCol="IS_ONLINE")

# # Evaluate the models
# accuracy_original = evaluator.evaluate(predictions_original)
# accuracy_pca = evaluator.evaluate(predictions_pca)

# # Print the evaluation results
# print(f"Accuracy of SVM on original data: {accuracy_original}")
# print(f"Accuracy of SVM on PCA data: {accuracy_pca}")


Training set size (original): 8000698


Test set size (original): 1999302


Training set size (PCA): 8000698


Test set size (PCA): 1999302


24/11/14 11:31:01 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:31:55 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:31:59 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:33:32 WARN MemoryStore: Not enough space to cache rdd_586_0 in memory! (computed 113.0 MiB so far)
24/11/14 11:33:32 WARN BlockManager: Persisting block rdd_586_0 to disk instead.
24/11/14 11:33:50 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:33:58 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:34:04 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:34:15 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:34:27 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:34:38 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:34:5

Accuracy of SVM on original data: 0.9955592854409048
Accuracy of SVM on PCA data: 0.9955682896304093


In [15]:
# # Assemble features for PCA data
# vector_assembler_pca = VectorAssembler(inputCols=["pca_features"], outputCol="pca_features_vector")
# data_with_pca_features = vector_assembler_pca.transform(result)

# # Split the data into training and test sets (80% train, 20% test)
# train_data, test_data = data_with_features.randomSplit([0.8, 0.2], seed=42)
# train_data_pca, test_data_pca = data_with_pca_features.randomSplit([0.8, 0.2], seed=42)

# # Print the size of the training and test sets
# print(f"Training set size (original): {train_data.count()}")
# print(f"Test set size (original): {test_data.count()}")
# print(f"Training set size (PCA): {train_data_pca.count()}")
# print(f"Test set size (PCA): {test_data_pca.count()}")

Training set size (original): 8000698


Test set size (original): 1999302


Training set size (PCA): 8000698


Test set size (PCA): 1999302


In [16]:
# Initialize LinearSVC models for original and PCA data
svm_original = LinearSVC(featuresCol="features", labelCol="IS_ONLINE")
svm_pca = LinearSVC(featuresCol="pca_features_vector", labelCol="IS_ONLINE")

# Train SVM on original data
svm_model_original = svm_original.fit(train_data)

# Train SVM on PCA data
svm_model_pca = svm_pca.fit(train_data_pca)

# Make predictions on test data
predictions_original_test = svm_model_original.transform(test_data)
predictions_pca_test = svm_model_pca.transform(test_data_pca)

# Make predictions on training data
predictions_original_train = svm_model_original.transform(train_data)
predictions_pca_train = svm_model_pca.transform(train_data_pca)


24/11/14 11:52:11 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:53:19 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:53:23 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:54:57 WARN MemoryStore: Not enough space to cache rdd_1877_0 in memory! (computed 113.0 MiB so far)
24/11/14 11:54:57 WARN BlockManager: Persisting block rdd_1877_0 to disk instead.
24/11/14 11:55:18 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:55:25 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:55:32 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:55:42 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:55:55 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:56:07 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/11/14 11:56

In [17]:
# Define a function for computing metrics
def compute_metrics(predictions, label_col="IS_ONLINE"):
    # Convert predictions to Pandas for evaluation
    predictions_pd = predictions.select("prediction", label_col).toPandas()
    y_true = predictions_pd[label_col]
    y_pred = predictions_pd["prediction"]

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, precision, recall, f1, conf_matrix


In [18]:
# # Evaluate models on training and test data
# metrics_original_train = compute_metrics(predictions_original_train)
# metrics_original_test = compute_metrics(predictions_original_test)
# metrics_pca_train = compute_metrics(predictions_pca_train)
# metrics_pca_test = compute_metrics(predictions_pca_test)

# # Print results for original data
# print("\nOriginal Data:")
# print(f"Training Data - Accuracy: {metrics_original_train[0]}, Precision: {metrics_original_train[1]}, Recall: {metrics_original_train[2]}, F1-Score: {metrics_original_train[3]}")
# print(f"Training Confusion Matrix:\n{metrics_original_train[4]}")
# print(f"Test Data - Accuracy: {metrics_original_test[0]}, Precision: {metrics_original_test[1]}, Recall: {metrics_original_test[2]}, F1-Score: {metrics_original_test[3]}")
# print(f"Test Confusion Matrix:\n{metrics_original_test[4]}")

# # Print results for PCA data
# print("\nPCA Data:")
# print(f"Training Data - Accuracy: {metrics_pca_train[0]}, Precision: {metrics_pca_train[1]}, Recall: {metrics_pca_train[2]}, F1-Score: {metrics_pca_train[3]}")
# print(f"Training Confusion Matrix:\n{metrics_pca_train[4]}")
# print(f"Test Data - Accuracy: {metrics_pca_test[0]}, Precision: {metrics_pca_test[1]}, Recall: {metrics_pca_test[2]}, F1-Score: {metrics_pca_test[3]}")
# print(f"Test Confusion Matrix:\n{metrics_pca_test[4]}")

# # Optionally, stop the Spark session
# spark.stop()


Original Data:
Training Data - Accuracy: 0.9935166906687392, Precision: 1.0, Recall: 0.9934529194847054, F1-Score: 0.9967157084818502
Training Confusion Matrix:
[[  77930       0]
 [  51871 7870897]]
Test Data - Accuracy: 0.9934532151720951, Precision: 1.0, Recall: 0.9933886994189288, F1-Score: 0.9966833861439074
Test Confusion Matrix:
[[  19510       0]
 [  13089 1966703]]

PCA Data:
Training Data - Accuracy: 0.9935166906687392, Precision: 1.0, Recall: 0.9934529194847054, F1-Score: 0.9967157084818502
Training Confusion Matrix:
[[  77930       0]
 [  51871 7870897]]
Test Data - Accuracy: 0.9934532151720951, Precision: 1.0, Recall: 0.9933886994189288, F1-Score: 0.9966833861439074
Test Confusion Matrix:
[[  19510       0]
 [  13089 1966703]]


In [4]:
ip_data_path = "/Users/yashwanthp/Downloads/country.csv"

# Read Parquet file into DataFrame
ip_data = spark.read.csv(ip_data_path,sep=",",header=True)

In [5]:
ip_data.show()

+---------+-----------+-------+------------+---------+--------------+
| start_ip|     end_ip|country|country_name|continent|continent_name|
+---------+-----------+-------+------------+---------+--------------+
|  1.0.0.0|  1.0.0.255|     AU|   Australia|       OC|       Oceania|
|  1.0.1.0|  1.0.3.255|     CN|       China|       AS|          Asia|
|  1.0.4.0|  1.0.7.255|     AU|   Australia|       OC|       Oceania|
|  1.0.8.0| 1.0.15.255|     CN|       China|       AS|          Asia|
| 1.0.16.0| 1.0.31.255|     JP|       Japan|       AS|          Asia|
| 1.0.32.0| 1.0.63.255|     CN|       China|       AS|          Asia|
| 1.0.64.0|1.0.127.255|     JP|       Japan|       AS|          Asia|
|1.0.128.0|1.0.255.255|     TH|    Thailand|       AS|          Asia|
|  1.1.0.0|  1.1.0.255|     CN|       China|       AS|          Asia|
|  1.1.1.0|  1.1.1.255|     AU|   Australia|       OC|       Oceania|
|  1.1.2.0|   1.1.6.21|     CN|       China|       AS|          Asia|
| 1.1.6.22|   1.1.6.

In [13]:
ip_data.count()

1938398

In [6]:
ns_data_path = "/Users/yashwanthp/Downloads/DAEN690_Data/indo_ns_full.csv"
columns = ["NS_ID", "ZONE_ID", "NAME", "IP"]
ns_data = spark.read.csv(ns_data_path,sep="\t",header=False)
ns_data = ns_data.toDF(*columns)

In [7]:
ns_data.show()

+-----+-------+-------------------+-------------+
|NS_ID|ZONE_ID|               NAME|           IP|
+-----+-------+-------------------+-------------+
|    1|  94175|a.gtld-servers.net.|   192.5.6.30|
|    2|  94175|b.gtld-servers.net.| 192.33.14.30|
|    3|  94175|c.gtld-servers.net.| 192.26.92.30|
|    4|  94175|d.gtld-servers.net.| 192.31.80.30|
|    5|  94175|e.gtld-servers.net.| 192.12.94.30|
|    6|  94175|f.gtld-servers.net.| 192.35.51.30|
|    7|  94175|g.gtld-servers.net.| 192.42.93.30|
|    8|  94175|h.gtld-servers.net.|192.54.112.30|
|    9|  94175|i.gtld-servers.net.|192.43.172.30|
|   10|  94175|j.gtld-servers.net.| 192.48.79.30|
|   11|  94175|k.gtld-servers.net.|192.52.178.30|
|   12|  94175|l.gtld-servers.net.|192.41.162.30|
|   13|  94175|m.gtld-servers.net.| 192.55.83.30|
|   14|  94181|a.gtld-servers.net.|   192.5.6.30|
|   15|  94181|b.gtld-servers.net.| 192.33.14.30|
|   16|  94181|c.gtld-servers.net.| 192.26.92.30|
|   17|  94181|d.gtld-servers.net.| 192.31.80.30|


In [21]:
def ip_to_long(ip):
    parts = ip.split(".")
    return int(parts[0]) * 256**3 + int(parts[1]) * 256**2 + int(parts[2]) * 256 + int(parts[3])

ip_to_long_udf = udf(ip_to_long, LongType())

In [27]:
ip_data = ip_data.filter(
    col("start_ip").rlike(r"^\d{1,3}(\.\d{1,3}){3}$") &
    col("end_ip").rlike(r"^\d{1,3}(\.\d{1,3}){3}$") &
    col("start_ip").isNotNull() &
    col("end_ip").isNotNull()
)

# Proceed with the rest of the workflow
ip_data = (
    ip_data
    .withColumn("start_ip_numeric", ip_to_long_udf("start_ip"))
    .withColumn("end_ip_numeric", ip_to_long_udf("end_ip"))
)

In [28]:
ns_data = ns_data.withColumn("ip_numeric", ip_to_long_udf("IP"))

In [29]:
# Join ns_data with ip_data
joined_data = ns_data.join(
    ip_data,
    (ns_data["ip_numeric"] >= ip_data["start_ip_numeric"]) &
    (ns_data["ip_numeric"] <= ip_data["end_ip_numeric"]),
    "inner"
)

In [32]:
joined_data.select(
    "NS_ID", "ZONE_ID", "NAME", "IP",
    "country", "country_name", "continent", "continent_name"
).show()

24/11/14 15:51:51 ERROR PythonUDFRunner: Python worker exited unexpectedly (crashed)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/Users/yashwanthp/anaconda3/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 1225, in main
    eval_type = read_int(infile)
                ^^^^^^^^^^^^^^^^
  File "/Users/yashwanthp/anaconda3/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 596, in read_int
    raise EOFError
EOFError

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:94)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:75)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(Int

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/var/folders/_d/94y4pt1163v0fnrmxtc1xgmr0000gn/T/ipykernel_74336/2249726730.py", line 3, in ip_to_long
ValueError: invalid literal for int() with base 10: 'NULL'


In [8]:
from pyspark.sql.functions import col

# Filter to exclude rows with 'NULL' as a string and keep only valid IPv4 addresses
ip_data = ip_data.filter(
    col("start_ip").rlike(r"^\d{1,3}(\.\d{1,3}){3}$") &
    col("end_ip").rlike(r"^\d{1,3}(\.\d{1,3}){3}$") &
    (col("start_ip") != 'NULL') &
    (col("end_ip") != 'NULL')  # Exclude rows where 'NULL' is a string
)

# UDF to convert IP to numeric
def ip_to_long(ip):
    if ip == 'NULL' or ip is None:
        return None  # Handle 'NULL' or None explicitly
    parts = ip.split(".")
    return int(parts[0]) * 256**3 + int(parts[1]) * 256**2 + int(parts[2]) * 256 + int(parts[3])

ip_to_long_udf = udf(ip_to_long, LongType())

# Apply the IP conversion to numeric values
ip_data = ip_data.withColumn("start_ip_numeric", ip_to_long_udf("start_ip"))
ip_data = ip_data.withColumn("end_ip_numeric", ip_to_long_udf("end_ip"))

# Filter out rows with invalid start_ip_numeric or end_ip_numeric
ip_data = ip_data.filter(col("start_ip_numeric").isNotNull() & col("end_ip_numeric").isNotNull())

# Apply the IP conversion to ns_data as well
ns_data = ns_data.withColumn("ip_numeric", ip_to_long_udf("IP"))

# Filter out rows with invalid IPs in ns_data
ns_data = ns_data.filter(col("ip_numeric").isNotNull())

# Join the two datasets
joined_data = ns_data.join(
    ip_data,
    (ns_data["ip_numeric"] >= ip_data["start_ip_numeric"]) &
    (ns_data["ip_numeric"] <= ip_data["end_ip_numeric"]),
    "inner"
)

# Select and show the relevant columns
joined_data.select(
    "NS_ID", "ZONE_ID", "NAME", "IP",
    "country", "country_name", "continent", "continent_name"
).show()


+-----+--------+----------------+---------+-------+------------+---------+--------------+
|NS_ID| ZONE_ID|            NAME|       IP|country|country_name|continent|continent_name|
+-----+--------+----------------+---------+-------+------------+---------+--------------+
|19248| 6401992|a.zdnscloud.com.|1.8.240.1|     CN|       China|       AS|          Asia|
|19249| 6401992|b.zdnscloud.com.|1.8.241.1|     CN|       China|       AS|          Asia|
|19250| 6401992|c.zdnscloud.com.|1.8.242.1|     CN|       China|       AS|          Asia|
|19251| 6401992|d.zdnscloud.com.|1.8.243.1|     CN|       China|       AS|          Asia|
|19525| 4234399|a.zdnscloud.com.|1.8.240.1|     CN|       China|       AS|          Asia|
|19526| 4234399|b.zdnscloud.com.|1.8.241.1|     CN|       China|       AS|          Asia|
|19527| 4234399|c.zdnscloud.com.|1.8.242.1|     CN|       China|       AS|          Asia|
|19528| 4234399|d.zdnscloud.com.|1.8.243.1|     CN|       China|       AS|          Asia|
|19665|290

24/11/14 20:56:17 WARN PythonUDFRunner: Detected deadlock while completing task 0.0 in stage 4 (TID 4): Attempting to kill Python Worker
24/11/14 20:56:17 WARN PythonUDFRunner: Detected deadlock while completing task 0.0 in stage 4 (TID 4): Attempting to kill Python Worker


In [9]:
# Get distinct IP values from the joined_data DataFrame
unique_ips = joined_data.select("IP").distinct()

# Show all unique IPs (optional - for small datasets)
unique_ips.show()

# Count the number of unique IPs
unique_ip_count = unique_ips.count()

# Print the count of unique IPs
print(f"Number of unique IP values: {unique_ip_count}")


ERROR:root:KeyboardInterrupt while sending command.              (2 + 8) / 1384]
Traceback (most recent call last):
  File "/Users/yashwanthp/anaconda3/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/yashwanthp/anaconda3/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/yashwanthp/anaconda3/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [10]:
data_path = "/Users/yashwanthp/Downloads/DAEN690_Data/indonesia_ips.csv"

indo_ips = spark.read.csv(data_path,sep=",",header=False)

indo_ips.count()

ERROR:root:KeyboardInterrupt while sending command.                 (0 + 0) / 1]
Traceback (most recent call last):
  File "/Users/yashwanthp/anaconda3/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/yashwanthp/anaconda3/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/yashwanthp/anaconda3/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 